In [39]:
#RESET OBJECT (copy from GITHUB)
import os
os.chdir('/content/')
!cd /content
!pwd
!ls -l | grep drw
!mkdir -p /content/bvarta_distribute
!rm -rf /content/bvarta_distribute/*
!rm -rf /content/bvarta_distribute/.git
!git clone https://github.com/tegarmaji/bvarta_distribute bvarta_distribute/
!ls -l /content/bvarta_distribute/

/content
drwxr-xr-x 4 root root 4096 Apr 23 04:08 bvarta_distribute
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data
Cloning into 'bvarta_distribute'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 27 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 9.96 KiB | 9.96 MiB/s, done.
Resolving deltas: 100% (4/4), done.
total 20
-rw-r--r-- 1 root root  160 Apr 23 04:08 config.yaml
drwxr-xr-x 4 root root 4096 Apr 23 04:08 data
-rw-r--r-- 1 root root 7871 Apr 23 04:08 pipeline.py
-rw-r--r-- 1 root root 2855 Apr 23 04:08 README.md


In [41]:
## RUN Pipeline
import os
os.chdir('/content/bvarta_distribute/')
!python pipeline.py --config config.yaml


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 04:08:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/23 04:08:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/events/day_*.jsonl.
java.io.FileNotFoundException: File data/raw/events/day_*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.F

------

In [42]:
#Check generated folder & data under Output folder
!ls -l /content/bvarta_distribute/output/*


/content/bvarta_distribute/output/bronze:
total 8
drwxr-xr-x 5 root root 4096 Apr 23 04:08 clean
drwxr-xr-x 2 root root 4096 Apr 23 04:08 quarantine

/content/bvarta_distribute/output/gold:
total 12
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2024-12-31'
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2025-01-01'
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2025-01-02'

/content/bvarta_distribute/output/silver:
total 12
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2024-12-31'
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2025-01-01'
drwxr-xr-x 2 root root 4096 Apr 23 04:09 'event_date=2025-01-02'


In [44]:
## Check Parquet output using pandas
import pandas as pd
import os
os.chdir('/content/bvarta_distribute/')
df_bronze_clean = pd.read_parquet(r'output/bronze/clean/')
df_bronze_quarantine = pd.read_parquet(r'output/bronze/quarantine/')
df_silver = pd.read_parquet(r'output/silver/')
df_gold = pd.read_parquet(r'output/gold/')


In [46]:
df_bronze_clean

,event_id,user_id,event_type,event_ts,value,event_date
0,e13,u2,VIEW,2024-12-31 23:50:00,1.0,2024-12-31
1,e1,u1,CLICK,2025-01-01 10:00:00,3.0,2025-01-01
2,e2,u2,VIEW,2025-01-01 10:05:00,0.0,2025-01-01
3,e3,u1,PURCHASE,2025-01-01 10:10:00,25.0,2025-01-01
4,e5,u3,CLICK,2025-01-01 11:00:00,1.0,2025-01-01
5,e6,u1,CLICK,2025-01-01 11:05:00,2.0,2025-01-01
6,e8,u2,PURCHASE,2025-01-01 11:30:00,30.0,2025-01-01
7,e9,u1,VIEW,2025-01-01 12:00:00,0.0,2025-01-01
8,e10,u1,CLICK,2025-01-02 09:00:00,1.0,2025-01-02
9,e11,u2,VIEW,2025-01-02 09:10:00,2.0,2025-01-02


In [47]:
df_bronze_quarantine

,event_id,user_id,event_type,event_ts,value,rejection_reason
0,e4,None,CLICK,invalid_ts,1,missing_user_id
1,e7,u2,VIEW,None,4,invalid_event_ts
2,e14,,CLICK,2025-01-02T10:00:00Z,1,missing_user_id
3,e15,u1,None,2025-01-02T10:05:00Z,3,missing_event_type
4,e16,u1,PURCHASE,2025-13-01T00:00:00Z,5,invalid_event_ts


In [48]:
df_silver

,user_id,event_id,event_type,event_ts,value,country,signup_date,is_purchase,days_since_signup,event_date
0,u2,e13,VIEW,2024-12-31 23:50:00,1.0,US,2024-12-15,False,16.0,2024-12-31
1,u1,e1,CLICK,2025-01-01 10:00:00,3.0,ID,2024-12-01,False,31.0,2025-01-01
2,u2,e2,VIEW,2025-01-01 10:05:00,0.0,US,2024-12-15,False,17.0,2025-01-01
3,u1,e3,PURCHASE,2025-01-01 10:10:00,25.0,ID,2024-12-01,True,31.0,2025-01-01
4,u3,e5,CLICK,2025-01-01 11:00:00,1.0,SG,None,False,NaN,2025-01-01
5,u1,e6,CLICK,2025-01-01 11:05:00,2.0,ID,2024-12-01,False,31.0,2025-01-01
6,u2,e8,PURCHASE,2025-01-01 11:30:00,30.0,US,2024-12-15,True,17.0,2025-01-01
7,u1,e9,VIEW,2025-01-01 12:00:00,0.0,ID,2024-12-01,False,31.0,2025-01-01
8,u1,e10,CLICK,2025-01-02 09:00:00,1.0,ID,2024-12-01,False,32.0,2025-01-02
9,u2,e11,VIEW,2025-01-02 09:10:00,2.0,US,2024-12-15,False,18.0,2025-01-02


In [49]:
df_gold

,country,total_events,total_value,total_purchases,unique_users,event_date
0,US,1,1.0,0,1,2024-12-31
1,SG,1,1.0,0,1,2025-01-01
2,ID,4,30.0,1,1,2025-01-01
3,US,2,30.0,1,1,2025-01-01
4,US,1,2.0,0,1,2025-01-02
5,ID,2,21.0,1,1,2025-01-02
